<div dir="rtl">
<h1>وقتی تعداد ویژگی‌ها امتیاز را بزرگ می‌کند</h1>
<p>درس 36 از 76 · چرا بر ریشهٔ تعداد ویژگی‌ها تقسیم می‌کنیم؟ · <code dir="ltr">30-scaling</code></p>
<p><a target="_self" href="http://127.0.0.1:8000/part-05/chapter-02/30-scaling.html">📖 بازگشت به همین درس</a></p>
<p>مقیاس درست امتیاز را پیاده کنید و ادعای آماری آن را از یک برابری دقیق جدا نگه دارید.</p><p>پیش‌نیاز: ضرب داخلی، Softmax و Standard deviation را از متن درس مرور کنید.</p>
<p>این دفتر نیمهٔ عملی درس است. مثال‌ها آمادهٔ اجرا هستند؛ دو Cell با برچسب TODO را خودتان کامل کنید. پیام INCOMPLETE یعنی هنوز چیزی ننوشته‌اید، نه اینکه پاسخ درست است. جواب مرجع در این دفتر پنهان نشده است.</p>
<p>از بالا به پایین اجرا کنید. پس از تغییر هر تابع، Cell آن و سپس Cell آزمون را دوباره اجرا کنید. برای بررسی نهایی، از منوی <code>Kernel → Restart Kernel and Run All Cells</code> استفاده کنید.</p>
</div>

In [ ]:
from pathlib import Path
import os
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "mini_gpt").is_dir() and (p / "book_src").is_dir()), None)
if project_root is None:
    raise RuntimeError("Extract the complete learning project; open this notebook inside it.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Python:", sys.executable)
print("Project:", project_root)

<div dir="rtl">
<h2>قبل از اجرا، پیش‌بینی کنید</h2>
<p>اگر پراکندگی امتیاز خام برای D=64 نزدیک ۸ باشد، تقسیم بر ۸ و تقسیم بر ۶۴ چه تفاوتی دارند؟</p>
</div>

<div dir="rtl"><p>پیش‌بینی من: …</p></div>

In [ ]:
import math
import torch
torch.set_num_threads(1)
torch.manual_seed(17)
q = torch.tensor([[1.,2.,3.,4.],[0.,1.,0.,1.]])
k = torch.tensor([[2.,0.,1.,0.],[1.,1.,1.,1.],[0.,0.,2.,2.]])
print('raw scores:',q@k.T)

<div dir="rtl">
<h2>این بار شما کد بنویسید</h2>
<p>تابع scaled_scores(q, k) را برای محور آخرِ ویژگی بنویسید. مخرج باید ریشهٔ تعداد ویژگی‌های همان بردار Query باشد؛ تعداد موقعیت‌ها یا C کل مدل را به تابع تحمیل نکنید.</p>
</div>

In [ ]:
def scaled_scores(q, k):
    # TODO
    return None

In [ ]:
def test_exercise():
    result = scaled_scores(q,k)
    if result is None: return False
    torch.testing.assert_close(result,(q@k.T)/2)
    a,b = torch.ones(2,3,9),torch.ones(2,5,9)
    torch.testing.assert_close(scaled_scores(a,b),torch.full((2,3,5),3.))
    torch.testing.assert_close(scaled_scores(torch.ones(1,1),torch.ones(2,1)),torch.ones(1,2))
    return True

exercise_complete = test_exercise()
print("PASS" if exercise_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl">
<h2>فقط یک عامل را تغییر دهید</h2>
<p>فقط D را تغییر دهید؛ ۲۰۰۰ جفت مستقل نرمال برای هر اندازه بسازید. این گزارش نمونه‌ای است: انتظار مقدار دقیقاً یک یا روند کاملاً یکنواخت نداریم.</p>
</div>

In [ ]:
generator = torch.Generator().manual_seed(31)
for D in (4,16,64):
    a = torch.randn(2000,D,generator=generator)
    b = torch.randn(2000,D,generator=generator)
    raw = (a*b).sum(-1)
    print(D,'raw std:',raw.std().item(),'scaled std:',(raw/math.sqrt(D)).std().item())

<div dir="rtl">
<h2>خرابی را پیدا کنید</h2>
<p>در Multi-Head Attention C=12 و H=3، هر بردار چهار ویژگی دارد. کد خراب بر sqrt(C) تقسیم می‌کند. تابع head_scores(q,k) را از خود Shape اصلاح کنید.</p>
</div>

In [ ]:
head_q,head_k = torch.ones(1,3,2,4),torch.ones(1,3,2,4)
print('wrong C scaling:',(head_q@head_k.transpose(-2,-1))/math.sqrt(12))

<div dir="rtl">
<h2>اصلاح را خودتان بنویسید</h2>
<p>علت را توضیح دهید، سپس تابع زیر را کامل کنید. خطای عمدی بالا یک نمونهٔ آموزشی است؛ آزمون پایین باید اصلاح شما را بسنجد.</p>
</div>

In [ ]:
def head_scores(q, k):
    # TODO
    return None

In [ ]:
def test_repair():
    result = head_scores(head_q,head_k)
    if result is None: return False
    torch.testing.assert_close(result,torch.full((1,3,2,2),2.))
    torch.testing.assert_close(head_scores(torch.ones(1,2,3,9),torch.ones(1,2,4,9)),torch.full((1,2,3,4),3.))
    return True

repair_complete = test_repair()
print("PASS" if repair_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl">
<h2>در Mini-GPT کجا به کار می‌آید؟</h2>
<p>CausalSelfAttention از head_dim استفاده می‌کند، نه channels. آزمون‌های دقیق ما خود تقسیم را می‌سنجند؛ آزمایش آماری فقط انگیزهٔ انتخاب آن را نشان می‌دهد.</p>
</div>

<div dir="rtl">
<h2>با زبان خودتان توضیح دهید</h2>
<p>کدام نتیجهٔ این دفتر یک قرارداد قطعی کد بود و کدام نتیجه به فرض‌های توزیع تصادفی وابسته بود؟</p>
</div>
<div dir="rtl"><p>پیش‌بینی و مشاهدهٔ من: …</p><p>علت خرابی و اصلاح من: …</p></div>

<div dir="rtl"><p><a target="_self" href="http://127.0.0.1:8000/part-05/chapter-02/30-scaling.html">بازگشت به درس و ادامهٔ مسیر</a> · <a target="_self" href="http://127.0.0.1:8000/answers/30-scaling.html#lab-solution">فقط پس از تلاش: راه‌حل مرجع آزمایشگاه</a></p></div>